# sWARm Future Projections - Overview

Generate 1-3 year WAR projections using longitudinal and survival modeling.

**Purpose:** Quick projection generation for league-wide future performance forecasting.

**Output:** `predictions/future_projections_hitter_YYYY.csv` and `predictions/future_projections_pitcher_YYYY.csv`

In [ ]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
project_root = Path('.').absolute()
if project_root.name == 'notebooks':
    project_root = project_root.parent.parent
sys.path.insert(0, str(project_root))

# Import future projection pipeline
from new_pipeline.models.future_season import (
    FutureProjectionPipeline,
    generate_league_projections
)

print("sWARm Future Projection System")
print("=" * 70)
print("Multi-year WAR projections using longitudinal & survival modeling")
print()

# Configuration
BASE_YEAR = 2024      # Year to project FROM
YEARS_AHEAD = 3       # Project 1, 2, 3 years ahead

print(f"Configuration:")
print(f"  Base year: {BASE_YEAR}")
print(f"  Projection years: {BASE_YEAR + 1} - {BASE_YEAR + YEARS_AHEAD}")
print(f"  Historical data: {BASE_YEAR - 8} - {BASE_YEAR}")

## Option 1: Quick League-Wide Projections (Recommended)

Generate projections for all hitters and pitchers with automatic constraint enforcement.

In [ ]:
# Cell 2: Generate League-Wide Projections

print("\nGenerating league-wide projections...")
print("This may take 5-10 minutes depending on data size.")
print()

# Generate projections for entire league (hitters + pitchers)
# Automatically applies zero-sum constraint (1000 WAR total)
hitter_projections, pitcher_projections = generate_league_projections(
    base_year=BASE_YEAR,
    years_ahead=YEARS_AHEAD,
    injury_records=None,  # Optional: pass injury data if available
    save_output=True
)

print("\nProjection generation complete!")
print(f"  Hitters projected: {len(hitter_projections)}")
print(f"  Pitchers projected: {len(pitcher_projections)}")

## Model Features (Enhanced for Year-to-Year Prediction)

**Pitchers (21 features):**
- Base features (11): BB%, K%, GB%, SwStr%, WPA/LI, Running_Control, Contact%, O-Swing%, Zone%, O-Contact%, F-Strike%
- Composites (5): strikeout_efficiency, contact_management, strikeout_contact_quality, Statcast_Launch_Quality_Index, SD_MD_Net
- Injury features (5): injury history and recovery indicators

**Removed Features (Low Year-to-Year Correlation):**
- ERA (r=0.38) - Replaced by component stats
- LOB% (r=0.22) - Too much luck/noise
- HR/FB% (r=0.29) - Unstable year-to-year
- damage_control_ratio, Opportunity_Success - Built on removed features

**Added Features (High Year-to-Year Correlation):**
- Contact% (r=0.79), O-Swing% (r=0.79), Zone% (r=0.78), O-Contact% (r=0.77), F-Strike% (r=0.64)

**Hitters (19 features):**
- Base features (14): K%, BB%, AVG, OBP, SLG, GDP, Positional_WAR, Enhanced_Baserunning, Enhanced_Defense, ISO, GB%, HR/FB, Hard%, Pull%
- Injury features (5): injury history and recovery indicators

**Added Features:**
- ISO (r=0.73), GB% (r=0.76), HR/FB (r=0.73), Hard% (r=0.60-0.65), Pull% (r=0.70+)

**Expected Performance Improvement:**
- Baseline R²: 0.35 → Enhanced R²: 0.45-0.50
- Baseline RMSE: 1.58 WAR → Enhanced RMSE: 1.35-1.45 WAR

## Ensemble Model Architecture

The projection system uses an advanced ensemble model combining multiple algorithms:
- **XGBoost** (Darts): Gradient boosting with automatic lag features
- **RNN (GRU)**: Recurrent neural network for trajectory learning
- **ExtraTrees** (Darts): Ensemble tree model with lags
- **Fallback Model**: Random Forest for players with limited history

**Adaptive Weighting:**
- Veterans (5+ seasons): XGB=0.35, RNN=0.35, ExtraTrees=0.30
- Mid-career (3-4 seasons): XGB=0.45, RNN=0.25, ExtraTrees=0.30
- Short history (<3 seasons): Fallback model only

**Additional Adjustments:**
- Age curve adjustments (position-specific)
- Survival probability discounting (retirement risk)
- Injury recovery curves (Tommy John, ACL)

In [ ]:
# Cell 3: Top Projected Hitters (Year 1)

print("\nTop 20 Projected Hitters (Year 1)")
print("=" * 70)

top_hitters = hitter_projections.nlargest(20, 'war_year_1')[[
    'playerid', 'war_year_1', 'war_year_2', 'war_year_3'
]].copy()

top_hitters.columns = ['Player ID', 'Year 1 WAR', 'Year 2 WAR', 'Year 3 WAR']
print(top_hitters.to_string(index=False))

In [ ]:
# Cell 4: Top Projected Pitchers (Year 1)

print("\nTop 20 Projected Pitchers (Year 1)")
print("=" * 70)

top_pitchers = pitcher_projections.nlargest(20, 'war_year_1')[[
    'playerid', 'war_year_1', 'war_year_2', 'war_year_3'
]].copy()

top_pitchers.columns = ['Player ID', 'Year 1 WAR', 'Year 2 WAR', 'Year 3 WAR']
print(top_pitchers.to_string(index=False))

## Projection Summary Statistics

In [ ]:
# Cell 5: Summary Statistics

print("\nProjection Summary Statistics")
print("=" * 70)

# Hitter stats
print("\nHitters:")
for year in range(1, YEARS_AHEAD + 1):
    war_col = f'war_year_{year}'
    total_war = hitter_projections[war_col].sum()
    mean_war = hitter_projections[war_col].mean()
    std_war = hitter_projections[war_col].std()
    print(f"  Year {year}: Total = {total_war:.1f} WAR, Mean = {mean_war:.2f}, Std = {std_war:.2f}")

# Pitcher stats
print("\nPitchers:")
for year in range(1, YEARS_AHEAD + 1):
    war_col = f'war_year_{year}'
    total_war = pitcher_projections[war_col].sum()
    mean_war = pitcher_projections[war_col].mean()
    std_war = pitcher_projections[war_col].std()
    print(f"  Year {year}: Total = {total_war:.1f} WAR, Mean = {mean_war:.2f}, Std = {std_war:.2f}")

# League total
print("\nLeague Total (should be ~1000 WAR):")
for year in range(1, YEARS_AHEAD + 1):
    war_col = f'war_year_{year}'
    league_total = hitter_projections[war_col].sum() + pitcher_projections[war_col].sum()
    print(f"  Year {year}: {league_total:.1f} WAR")

## Option 2: Step-by-Step Pipeline (For Advanced Users)

Run hitter and pitcher pipelines separately with full control.

In [ ]:
# Cell 6: Manual Pipeline - Hitters

print("\nRunning hitter projection pipeline...")
hitter_pipeline = FutureProjectionPipeline(
    player_type='hitter',
    base_year=BASE_YEAR,
    years_ahead=YEARS_AHEAD
)

# Run full pipeline
hitter_projections_manual = hitter_pipeline.run_full_pipeline(
    injury_records=None,
    save_output=False  # Don't save yet
)

print(f"\nHitter projections complete: {len(hitter_projections_manual)} players")

In [ ]:
# Cell 7: Manual Pipeline - Pitchers

print("\nRunning pitcher projection pipeline...")
pitcher_pipeline = FutureProjectionPipeline(
    player_type='pitcher',
    base_year=BASE_YEAR,
    years_ahead=YEARS_AHEAD
)

# Run full pipeline
pitcher_projections_manual = pitcher_pipeline.run_full_pipeline(
    injury_records=None,
    save_output=False
)

print(f"\nPitcher projections complete: {len(pitcher_projections_manual)} players")

In [ ]:
# Cell 8: Apply League-Wide Constraints

print("\nApplying league-wide zero-sum constraints...")

# Apply constraints to enforce 1000 total WAR
hitter_pipeline.apply_constraints(
    hitter_projections=hitter_projections_manual,
    pitcher_projections=pitcher_projections_manual
)

pitcher_pipeline.apply_constraints(
    hitter_projections=hitter_projections_manual,
    pitcher_projections=pitcher_projections_manual
)

# Save outputs
hitter_pipeline.save_projections()
pitcher_pipeline.save_projections()

print("\nProjections saved to predictions/ directory")

## Done!

Projections have been generated and saved to:
- `predictions/future_projections_hitter_YYYY.csv`
- `predictions/future_projections_pitcher_YYYY.csv`

For detailed backtesting and error analysis, see `sWARm_future_deep_dive.ipynb`.